[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/72_non_max_suppression_solution.ipynb)

# 🟡 Solution: Non-Max Suppression

Reference solution for `non_max_suppression`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def _pairwise_iou_one(box: torch.Tensor, boxes: torch.Tensor) -> torch.Tensor:
    lt = torch.maximum(box[:2], boxes[:, :2])
    rb = torch.minimum(box[2:], boxes[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, 0] * wh[:, 1]
    area1 = ((box[2] - box[0]).clamp(min=0) * (box[3] - box[1]).clamp(min=0))
    area2 = ((boxes[:, 2] - boxes[:, 0]).clamp(min=0) * (boxes[:, 3] - boxes[:, 1]).clamp(min=0))
    union = area1 + area2 - inter
    return inter / union.clamp_min(1e-7)


def non_max_suppression(boxes: torch.Tensor, scores: torch.Tensor,
                        iou_threshold: float = 0.5) -> torch.Tensor:
    order = torch.argsort(scores, descending=True)
    keep = []
    while order.numel() > 0:
        i = order[0]
        keep.append(i)
        if order.numel() == 1:
            break
        rest = order[1:]
        ious = _pairwise_iou_one(boxes[i], boxes[rest])
        order = rest[ious <= iou_threshold]
    if not keep:
        return torch.empty(0, dtype=torch.long, device=boxes.device)
    return torch.stack(keep).to(torch.long)


In [ ]:
# Verify
boxes = torch.tensor([[0.,0.,2.,2.], [0.2,0.2,2.2,2.2], [5.,5.,6.,6.]])
scores = torch.tensor([0.9, 0.8, 0.7])
print(non_max_suppression(boxes, scores, 0.5))


In [ ]:
# Run judge
from torch_judge import check
check('non_max_suppression')
